In [1]:
import pandas as pd
import numpy as np
import math
import folium
from folium.plugins import MeasureControl

In [2]:
ocean = pd.read_csv('/content/dataset/OceanQuality_Clustered_Results.csv')
land = pd.read_csv('/content/dataset/WaterQuality_Clustered_Results.csv')
metadata = pd.read_excel('/content/dataset/metadata.xlsx')

print("Đã tải dữ liệu thành công!")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/OceanQuality_Clustered_Results.csv'

In [3]:
land_with_meta = pd.merge(
    land,
    metadata[['MonitoringLocationIdentifier', 'MonitoringLocationName', 'MonitoringLocationTypeName', 'MonitoringLocationTypeCode', 'LatitudeMeasure_WGS84', 'LongitudeMeasure_WGS84']],
    on='MonitoringLocationIdentifier',
    how='inner'
)

land_with_meta = land_with_meta.rename(columns={
    'LatitudeMeasure_WGS84': 'Latitude',
    'LongitudeMeasure_WGS84': 'Longitude',
    'MonitoringLocationName': 'Station'
})

ocean_with_meta = pd.merge(
    ocean,
    metadata[['MonitoringLocationIdentifier', 'MonitoringLocationName', 'LatitudeMeasure_WGS84', 'LongitudeMeasure_WGS84']],
    on='MonitoringLocationIdentifier',
    how='inner'
)
ocean_with_meta = ocean_with_meta.rename(columns={
    'LatitudeMeasure_WGS84': 'Latitude',
    'LongitudeMeasure_WGS84': 'Longitude',
    'MonitoringLocationName': 'Station'
})
ocean_stations = ocean_with_meta.drop_duplicates(subset=['MonitoringLocationIdentifier']).copy()

NameError: name 'land' is not defined

In [4]:
all_river_stations = land_with_meta.drop_duplicates(subset=['MonitoringLocationIdentifier']).copy()


# Reuse 'ocean_stations' from previous steps


all_data_lats = pd.concat([all_river_stations['Latitude'], ocean_stations['Latitude']])
all_data_lons = pd.concat([all_river_stations['Longitude'], ocean_stations['Longitude']])

center_lat_all = all_data_lats.mean()
center_lon_all = all_data_lons.mean()

m_all = folium.Map(location=[center_lat_all, center_lon_all], zoom_start=5,
                   tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
                   attr='Google')

m_all.add_child(MeasureControl())

for _, row in all_river_stations.iterrows():
    info_text = f"""
    <div style=\"width: 200px;\">
        <b>Tên trạm:</b> {row['Station']}<br>
        <b>Mã trạm:</b> {row['MonitoringLocationIdentifier']}<br>
        <b>Loại:</b> Sông ({row['MonitoringLocationTypeName']})
    </div>
    """
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color='blue', icon='tint'),
        popup=folium.Popup(info_text, max_width=300),
        tooltip=row['Station']
    ).add_to(m_all)

for _, row in ocean_stations.iterrows():
    info_text = f"""
    <div style=\"width: 200px;\">
        <b>Mã trạm:</b> {row['MonitoringLocationIdentifier']}<br>
        <b>Tên trạm:</b> {row['Station']}<br>
        <b>Loại:</b> Biển
    </div>
    """
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        icon=folium.Icon(color='red', icon='fa-water', prefix='fa'),
        popup=folium.Popup(info_text, max_width=300),
        tooltip=row['MonitoringLocationIdentifier']
    ).add_to(m_all)

m_all

m_all.save('map.html')

NameError: name 'land_with_meta' is not defined